# nb_train_validate — 학습/검증 → 승인 모델 패키지

Gold(Delta) → 피처·표본 → Ridge 학습·보정 → 승인 게이트 → 패키지(+MANIFEST) → MLflow 기록

- 이 노트북에는 계산이 없습니다. `analytics/src`의 함수만 호출합니다.
- **빈 칸(`None`)은 Fabric/DevOps 담당과 확인해 채울 값입니다.** 비어 있으면 실행이 시작 전에 멈추고 빈 키를 알려 줍니다.
- 비밀번호·접속 문자열은 쓰지 않습니다. Key Vault 비밀 *이름*만 적습니다.
- 승인 기준 숫자가 비어 있으면 패키지는 `candidate`로만 만들어지고 MLflow 모델 등록은 하지 않습니다.

In [ ]:
# === 설정 칸 (여기만 수정) ===
CODE_DIR = None            # analytics/src를 올려둔 Lakehouse 경로 예: '/lakehouse/default/Files/analytics/src'
CFG = {
    # --- Gold 읽기 ---
    'gold_kind': 'delta_path',      # 'delta_path' | 'spark_table' | 'parquet'
    'gold_location': None,          # Delta 폴더 경로 또는 표 이름 (Fabric 담당 확인)
    'dataset_version': None,        # 예: 'official20-gold-v1.1-...'  (None이면 표에 버전이 하나일 때만 통과)
    'delta_version': None,          # 숫자를 주면 그 Delta 버전으로 고정해서 읽음 (권장: 기록용으로 채우기)
    'strict': False,                # True: 공개 시각 NULL 행을 오류 처리 (시점 검증 완료 후 사용)
    # --- 산출물 ---
    'work_dir': '/lakehouse/default/Files/analytics_runs/train_validate',
    'package_name': None,           # 예: 'ridge-2026-11-09' (같은 이름 재사용 불가)
    'criteria_path': None,          # 승인 기준 JSON. None이면 항상 candidate
    # --- MLflow (둘 다 채우면 기록. 모델 등록은 approved일 때만) ---
    'mlflow_experiment': None,
    'mlflow_model_name': None,
}
EXTRA_METRICS = {}          # 게이트 지표 중 노트북 밖에서 계산한 값 (예: {'ridge_migration_gap': 0.0})

In [ ]:
import sys, json
if CODE_DIR: sys.path.insert(0, CODE_DIR)
import run_release as RR
result = RR.run_train_validate(CFG, spark=spark, extra_metrics=EXTRA_METRICS)

In [ ]:
print(json.dumps({
    'package_id': result['package']['package_id'], 'status': result['package']['status'], 'dir': result['package']['dir'],
    'gate_approved': result['gate']['approved'], 'undefined_thresholds': result['gate']['undefined_thresholds'],
    'missing_required_checks': result['gate']['missing_required_checks'],
    'leak_checks': result['leak_checks'], 'mlflow': result['mlflow'],
    'gold': {k: result['gold'][k] for k in ('rows','dataset_versions','trading_date_max','assets')},
}, ensure_ascii=False, indent=2, default=str))

**다음 단계**: `status`가 `approved`이고 MLflow 등록이 끝난 패키지 폴더를 DevOps 파이프라인이 `verify_package`로 검증한 뒤 ACR 이미지에 포함합니다. `candidate`는 배포 대상이 아닙니다.